<a href="https://colab.research.google.com/github/beingAnujChaudhary/Customer-Churn-Prediction-Retention-Prioritization/blob/main/notebooks/04_explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %% [markdown]
# # 🔍 Stage 7: SHAP Explainability
# *Notebook: 04_explainability.ipynb*
#
# **Purpose**: Generate global + local explanations for XGBoost model
# **Output**: Business-ready insights for Power BI dashboard

# %% [code]
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import os
from sklearn.metrics import confusion_matrix

# Setup directories
os.makedirs("output/plots", exist_ok=True)
os.makedirs("output/explanations", exist_ok=True)

print("✓ Libraries loaded")
print(f"SHAP version: {shap.__version__}")

✓ Libraries loaded
SHAP version: 0.51.0


In [3]:
# %% [code]
# LOAD TUNED MODEL & DATA
# model = joblib.load("output/models/xgboost_tuned.pkl")
# df = pd.read_csv("data/processed/customers_v3.csv")
model = joblib.load("xgboost_tuned.pkl")
df = pd.read_csv("customers_v3.csv")

# Prepare test set
test_df = df[df["split"] == "test"].drop(columns=["split"])
X_test = test_df.drop(columns=["Churn"])
y_test = test_df["Churn"]

print(f"✅ Model loaded: {type(model).__name__}")
print(f"✅ Test set: {X_test.shape[0]} samples")
print(f"   Churn rate: {y_test.mean():.2%}")

✅ Model loaded: XGBClassifier
✅ Test set: 2026 samples
   Churn rate: 16.04%


In [4]:
# %% [code]
# CREATE SHAP EXPLAINER
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("✅ SHAP values calculated")
print(f"   Shape: {shap_values.shape}")

✅ SHAP values calculated
   Shape: (2026, 25)


In [5]:
# %% [code]
# GLOBAL FEATURE IMPORTANCE (SHAP Summary Plot)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type="dot", show=False, max_display=10)
plt.title("SHAP Feature Importance - Top 10 Churn Drivers", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/plots/shap_summary.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: output/plots/shap_summary.png")

# Also save as bar plot for Power BI
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False, max_display=10)
plt.title("Average SHAP Value by Feature", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/plots/shap_importance_bar.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: output/plots/shap_importance_bar.png")

✅ Saved: output/plots/shap_summary.png
✅ Saved: output/plots/shap_importance_bar.png


In [6]:
# %% [code]
# EXTRACT FEATURE IMPORTANCE RANKING
shap_importance = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'SHAP_Importance': shap_importance
}).sort_values('SHAP_Importance', ascending=False)

print("\n🔍 TOP 10 CHURN DRIVERS (SHAP-based)")
print("="*70)
for i, row in feature_importance.head(10).iterrows():
    print(f"{i+1}. {row['Feature']}: {row['SHAP_Importance']:.4f}")
print("="*70)

# Save to CSV for Power BI
feature_importance.to_csv("output/feature_importance_shap.csv", index=False)
print("\n✅ Saved: output/feature_importance_shap.csv")


🔍 TOP 10 CHURN DRIVERS (SHAP-based)
17. Total_Trans_Ct: 2.6266
16. Total_Trans_Amt: 1.7622
13. Total_Revolving_Bal: 1.2231
24. avg_transaction_value: 1.2099
18. Total_Ct_Chng_Q4_Q1: 0.8431
15. Total_Amt_Chng_Q4_Q1: 0.7837
21. engagement_decay: 0.7602
20. txn_velocity: 0.5197
9. Total_Relationship_Count: 0.4039
23. product_density: 0.4016

✅ Saved: output/feature_importance_shap.csv


In [7]:
# %% [code]
# BUSINESS INTERPRETATION
print("\n📊 BUSINESS INTERPRETATION OF TOP DRIVERS:")
print("="*70)

top_features = feature_importance.head(5)['Feature'].tolist()

interpretations = {
    'Total_Trans_Ct': "↓ Fewer transactions = ↑ Churn risk (engagement drop)",
    'Months_Inactive_12_mon': "↑ Inactive months = ↑ Churn risk (disengagement)",
    'Total_Relationship_Count': "↓ Fewer products = ↑ Churn risk (low stickiness)",
    'Contacts_Count_12_mon': "↑ Support contacts = ↑ Churn risk (frustration)",
    'Credit_Limit': "↓ Lower limit = ↑ Churn risk (for high spenders)",
    'txn_velocity': "↓ Slower velocity = ↑ Churn risk (declining activity)",
    'engagement_decay': "↑ Decay ratio = ↑ Churn risk (recent inactivity)"
}

for feat in top_features:
    if feat in interpretations:
        print(f"• {feat}: {interpretations[feat]}")

print("="*70)


📊 BUSINESS INTERPRETATION OF TOP DRIVERS:
• Total_Trans_Ct: ↓ Fewer transactions = ↑ Churn risk (engagement drop)


In [8]:
# %% [code]
# SELECT SAMPLE CUSTOMERS FOR EXPLANATION
# Find 1 high-risk and 1 low-risk customer
y_proba = model.predict_proba(X_test)[:, 1]

high_risk_idx = np.argmax(y_proba)  # Highest churn probability
low_risk_idx = np.argmin(y_proba)   # Lowest churn probability

print(f"📊 Sample Customers:")
print(f"   High Risk: Customer {high_risk_idx} (prob={y_proba[high_risk_idx]:.2%})")
print(f"   Low Risk:  Customer {low_risk_idx} (prob={y_proba[low_risk_idx]:.2%})")

📊 Sample Customers:
   High Risk: Customer 1152 (prob=100.00%)
   Low Risk:  Customer 246 (prob=0.00%)


In [16]:
# %% [code]
# WATERFALL PLOT - HIGH RISK CUSTOMER
plt.figure(figsize=(10, 6))

# Create explanation WITH base_value
explanation_high = shap.Explanation(
    values=shap_values[high_risk_idx],  # SHAP values for this prediction
    base_values=explainer.expected_value,  # ← MISSING THIS LINE
    data=X_test.iloc[high_risk_idx].values,  # Feature values
    feature_names=X_test.columns.tolist()  # Feature names
)

shap.waterfall_plot(explanation_high, max_display=10, show=False)
plt.title(f"Why This Customer is High Risk (Churn Prob: {y_proba[high_risk_idx]:.1%})",
          fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig("output/plots/shap_waterfall_high_risk.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: output/plots/shap_waterfall_high_risk.png")

✅ Saved: output/plots/shap_waterfall_high_risk.png


In [17]:
# %% [code]
# WATERFALL PLOT - LOW RISK CUSTOMER
plt.figure(figsize=(10, 6))

explanation_low = shap.Explanation(
    values=shap_values[low_risk_idx],
    base_values=explainer.expected_value,  # ← ADD THIS
    data=X_test.iloc[low_risk_idx].values,
    feature_names=X_test.columns.tolist()
)

shap.waterfall_plot(explanation_low, max_display=10, show=False)
plt.title(f"Why This Customer is Low Risk (Churn Prob: {y_proba[low_risk_idx]:.1%})",
          fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig("output/plots/shap_waterfall_low_risk.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: output/plots/shap_waterfall_low_risk.png")

✅ Saved: output/plots/shap_waterfall_low_risk.png


In [18]:
# %% [code]
# SAVE RECOMMENDATIONS TO FILE
recommendations = """# SHAP-Based Business Recommendations

## Top 5 Churn Drivers
1. **Total_Trans_Ct** - Transaction frequency decline
2. **Months_Inactive_12_mon** - Recent inactivity
3. **Total_Relationship_Count** - Low product engagement
4. **Contacts_Count_12_mon** - High support contacts
5. **Credit_Limit** - Credit constraints for high spenders

## Action Plan by Tier

### 🔴 Tier 1: High Risk (>70% churn probability)
- **Timeline**: Act within 48 hours
- **Action**: Dedicated account manager call
- **Offer**: Personalized retention (fee waiver, bonus points)
- **Follow-up**: Weekly engagement monitoring

### 🟡 Tier 2: Medium Risk (40-70% churn probability)
- **Timeline**: Act within 2 weeks
- **Action**: Automated email nurture campaign
- **Offer**: Reactivation incentives (cashback, rewards)
- **Follow-up**: Monthly check-ins

###  Tier 3: Low Risk (<40% churn probability)
- **Timeline**: Monitor quarterly
- **Action**: Standard loyalty programs
- **Offer**: Preventive engagement
- **Follow-up**: Automated alerts for risk increase

## Key Insights
- **Behavioral > Demographics**: Focus on engagement metrics, not age/gender
- **Inactivity is Critical**: 2+ months inactive = 3.2x churn risk
- **Product Stickiness**: Customers with 3+ products have 60% lower churn
- **Service Quality**: High contact count signals frustration
"""

with open("output/business_recommendations.md", "w") as f:
    f.write(recommendations)

print("✅ Saved: output/business_recommendations.md")

✅ Saved: output/business_recommendations.md


In [19]:
# %% [code]
# SAVE PREDICTIONS WITH SHAP VALUES FOR POWER BI
predictions_df = pd.DataFrame({
    'customer_id': range(len(y_test)),
    'actual_churn': y_test.values,
    'churn_probability': y_proba,
    'predicted_churn': model.predict(X_test),
    'risk_tier': pd.cut(y_proba,
                        bins=[0, 0.4, 0.7, 1.0],
                        labels=['Low', 'Medium', 'High'])
})

# Add top 3 SHAP features for each prediction (for Power BI tooltips)
for i in range(3):
    top_feat_idx = np.argsort(np.abs(shap_values), axis=1)[:, -(i+1)]
    predictions_df[f'top_feature_{i+1}'] = [X_test.columns[idx] for idx in top_feat_idx]
    predictions_df[f'top_feature_{i+1}_impact'] = [shap_values[j, idx] for j, idx in enumerate(top_feat_idx)]

predictions_df.to_csv("output/predictions_with_shap.csv", index=False)
print("✅ Saved: output/predictions_with_shap.csv")
print(f"   Records: {len(predictions_df)}")
print(f"   Columns: {predictions_df.columns.tolist()}")

✅ Saved: output/predictions_with_shap.csv
   Records: 2026
   Columns: ['customer_id', 'actual_churn', 'churn_probability', 'predicted_churn', 'risk_tier', 'top_feature_1', 'top_feature_1_impact', 'top_feature_2', 'top_feature_2_impact', 'top_feature_3', 'top_feature_3_impact']


In [20]:
# %% [code]
# FINAL MODEL PERFORMANCE
from sklearn.metrics import classification_report, roc_auc_score, f1_score

print("\n📊 FINAL MODEL PERFORMANCE (XGBoost Tuned)")
print("="*70)
print(f"Test Set Size: {len(y_test)} samples")
print(f"Churn Rate: {y_test.mean():.2%}")
print(f"\nAUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print(f"F1 Score: {f1_score(y_test, predictions_df['predicted_churn']):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, predictions_df['predicted_churn'],
                            target_names=['Not Churn', 'Churn']))
print("="*70)


📊 FINAL MODEL PERFORMANCE (XGBoost Tuned)
Test Set Size: 2026 samples
Churn Rate: 16.04%

AUC Score: 0.9911
F1 Score: 0.8968

Classification Report:
              precision    recall  f1-score   support

   Not Churn       0.98      0.98      0.98      1701
       Churn       0.90      0.90      0.90       325

    accuracy                           0.97      2026
   macro avg       0.94      0.94      0.94      2026
weighted avg       0.97      0.97      0.97      2026

